# Modélisation Baseline

Notebook 3 : Entraînement de deux modèles simples et comparaison de leurs performances

## Étape 1 : CHARGEMENT DES DONNÉES PRÉPARÉES

In [ ]:
# Imports des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import pickle
import warnings
warnings.filterwarnings('ignore')

# Configuration des plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("="*70)
print("MODÉLISATION BASELINE")
print("="*70)
print("\n[OK] Bibliothèques importées")

In [ ]:
# Charger les fichiers préparés
print("\n" + "="*70)
print("ÉTAPE 1 : CHARGEMENT DES DONNÉES")
print("="*70)

X = pd.read_csv('data/gold/X_features.csv')
y = pd.read_csv('data/gold/y_target.csv')

print(f"\n[OK] Données chargées")
print(f"  X (features) : {X.shape[0]:,} lignes x {X.shape[1]} colonnes")
print(f"  y (target)   : {y.shape[0]:,} lignes x {y.shape[1]} colonnes")

# Vérifier que les indices correspondent
if X.shape[0] != y.shape[0]:
    print(f"\n[WARNING] Les tailles de X et y ne correspondent pas!")
else:
    print(f"\n[OK] Les tailles de X et y correspondent")

In [ ]:
# Afficher les informations détaillées
print(f"\nColonnes de X (features) :")
for col in X.columns:
    print(f"  - {col}")

print(f"\nColonne de y (cible) :")
for col in y.columns:
    print(f"  - {col}")

print(f"\nStatistiques de X :")
print(X.describe())

print(f"\nStatistiques de y :")
print(y.describe())

## Étape 2 : SÉPARATION TRAIN/TEST

In [ ]:
from sklearn.model_selection import train_test_split

print("\n" + "="*70)
print("ÉTAPE 2 : SÉPARATION TRAIN/TEST")
print("="*70)

# Séparation train/test : 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Convertir y en vecteur 1D
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

print(f"\n[OK] Données séparées (80/20)")
print(f"  Train : {X_train.shape[0]:,} x {X_train.shape[1]}")
print(f"  Test  : {X_test.shape[0]:,} x {X_test.shape[1]}")
print(f"\n  y_train : {y_train.shape[0]:,}")
print(f"  y_test  : {y_test.shape[0]:,}")

In [ ]:
# Standardiser les features numériques
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n[OK] Features normalisées avec StandardScaler")
print(f"  Mean train (avant scaling) : {X_train.mean().mean():.2f}")
print(f"  Std train (avant scaling)  : {X_train.std().mean():.2f}")
print(f"  Mean train (après scaling) : {X_train_scaled.mean():.6f}")
print(f"  Std train (après scaling)  : {X_train_scaled.std():.6f}")

## Étape 3 : PREMIER MODÈLE - RÉGRESSION LINÉAIRE

In [ ]:
print("\n" + "="*70)
print("ÉTAPE 3 : RÉGRESSION LINÉAIRE")
print("="*70)

# Créer et entraîner le modèle
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

print(f"\n[OK] Modèle Régression Linéaire entraîné")
print(f"  Nombre de features : {lr_model.coef_.shape[0]}")
print(f"  Intercept : {lr_model.intercept_:.4f}")

# Prédictions
y_train_pred_lr = lr_model.predict(X_train_scaled)
y_test_pred_lr = lr_model.predict(X_test_scaled)

print(f"\n[OK] Prédictions effectuées")

In [ ]:
# Calculer les métriques
# Les prédictions sont en log(prix_m2), il faut les convertir en EUR/m²

def calculate_metrics(y_true, y_pred, set_name='Test'):
    """Calculer les métriques en log et en EUR/m²"""
    
    # En log
    mae_log = mean_absolute_error(y_true, y_pred)
    rmse_log = np.sqrt(mean_squared_error(y_true, y_pred))
    r2_log = r2_score(y_true, y_pred)
    
    # Convertir en EUR/m²
    y_true_euros = np.exp(y_true)
    y_pred_euros = np.exp(y_pred)
    
    mae_euros = mean_absolute_error(y_true_euros, y_pred_euros)
    rmse_euros = np.sqrt(mean_squared_error(y_true_euros, y_pred_euros))
    
    # MAPE (Mean Absolute Percentage Error)
    mape = np.mean(np.abs((y_true_euros - y_pred_euros) / y_true_euros)) * 100
    
    return {
        'mae_log': mae_log,
        'rmse_log': rmse_log,
        'r2_log': r2_log,
        'mae_euros': mae_euros,
        'rmse_euros': rmse_euros,
        'mape': mape
    }

metrics_lr_train = calculate_metrics(y_train, y_train_pred_lr, 'Train')
metrics_lr_test = calculate_metrics(y_test, y_test_pred_lr, 'Test')

print(f"\n[RÉGRESSION LINÉAIRE] - Métriques d'entraînement")
print(f"  MAE (log)      : {metrics_lr_train['mae_log']:.4f}")
print(f"  RMSE (log)     : {metrics_lr_train['rmse_log']:.4f}")
print(f"  R² (log)       : {metrics_lr_train['r2_log']:.4f}")
print(f"  MAE (EUR/m²)   : {metrics_lr_train['mae_euros']:,.0f} EUR")
print(f"  RMSE (EUR/m²)  : {metrics_lr_train['rmse_euros']:,.0f} EUR")
print(f"  MAPE           : {metrics_lr_train['mape']:.2f}%")

print(f"\n[RÉGRESSION LINÉAIRE] - Métriques de test")
print(f"  MAE (log)      : {metrics_lr_test['mae_log']:.4f}")
print(f"  RMSE (log)     : {metrics_lr_test['rmse_log']:.4f}")
print(f"  R² (log)       : {metrics_lr_test['r2_log']:.4f}")
print(f"  MAE (EUR/m²)   : {metrics_lr_test['mae_euros']:,.0f} EUR")
print(f"  RMSE (EUR/m²)  : {metrics_lr_test['rmse_euros']:,.0f} EUR")
print(f"  MAPE           : {metrics_lr_test['mape']:.2f}%")

## Étape 4 : DEUXIÈME MODÈLE - XGBOOST

In [ ]:
print("\n" + "="*70)
print("ÉTAPE 4 : XGBOOST")
print("="*70)

# Créer et entraîner le modèle XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

xgb_model.fit(X_train_scaled, y_train)

print(f"\n[OK] Modèle XGBoost entraîné")
print(f"  Paramètres :")
print(f"    - n_estimators : 100")
print(f"    - max_depth : 5")
print(f"    - learning_rate : 0.1")
print(f"    - subsample : 0.8")
print(f"    - colsample_bytree : 0.8")

In [ ]:
# Prédictions
y_train_pred_xgb = xgb_model.predict(X_train_scaled)
y_test_pred_xgb = xgb_model.predict(X_test_scaled)

print(f"\n[OK] Prédictions effectuées")

# Calculer les métriques
metrics_xgb_train = calculate_metrics(y_train, y_train_pred_xgb, 'Train')
metrics_xgb_test = calculate_metrics(y_test, y_test_pred_xgb, 'Test')

print(f"\n[XGBOOST] - Métriques d'entraînement")
print(f"  MAE (log)      : {metrics_xgb_train['mae_log']:.4f}")
print(f"  RMSE (log)     : {metrics_xgb_train['rmse_log']:.4f}")
print(f"  R² (log)       : {metrics_xgb_train['r2_log']:.4f}")
print(f"  MAE (EUR/m²)   : {metrics_xgb_train['mae_euros']:,.0f} EUR")
print(f"  RMSE (EUR/m²)  : {metrics_xgb_train['rmse_euros']:,.0f} EUR")
print(f"  MAPE           : {metrics_xgb_train['mape']:.2f}%")

print(f"\n[XGBOOST] - Métriques de test")
print(f"  MAE (log)      : {metrics_xgb_test['mae_log']:.4f}")
print(f"  RMSE (log)     : {metrics_xgb_test['rmse_log']:.4f}")
print(f"  R² (log)       : {metrics_xgb_test['r2_log']:.4f}")
print(f"  MAE (EUR/m²)   : {metrics_xgb_test['mae_euros']:,.0f} EUR")
print(f"  RMSE (EUR/m²)  : {metrics_xgb_test['rmse_euros']:,.0f} EUR")
print(f"  MAPE           : {metrics_xgb_test['mape']:.2f}%")

In [ ]:
# Comparaison des modèles
print("\n" + "="*70)
print("COMPARAISON DES MODÈLES")
print("="*70)

comparison = pd.DataFrame({
    'Métrique': ['MAE (EUR/m²)', 'RMSE (EUR/m²)', 'MAPE (%)', 'R² (log)'],
    'Régression Linéaire': [
        f"{metrics_lr_test['mae_euros']:,.0f}",
        f"{metrics_lr_test['rmse_euros']:,.0f}",
        f"{metrics_lr_test['mape']:.2f}",
        f"{metrics_lr_test['r2_log']:.4f}"
    ],
    'XGBoost': [
        f"{metrics_xgb_test['mae_euros']:,.0f}",
        f"{metrics_xgb_test['rmse_euros']:,.0f}",
        f"{metrics_xgb_test['mape']:.2f}",
        f"{metrics_xgb_test['r2_log']:.4f}"
    ]
})

print("\nPerformances sur l'ensemble de test :")
print(comparison.to_string(index=False))

# Identifier le meilleur modèle
if metrics_xgb_test['rmse_euros'] < metrics_lr_test['rmse_euros']:
    print("\n[OK] XGBoost est meilleur que la Régression Linéaire")
else:
    print("\n[OK] Régression Linéaire est meilleur que XGBoost")

## Étape 5 : IMPORTANCE DES VARIABLES

In [ ]:
print("\n" + "="*70)
print("ÉTAPE 5 : IMPORTANCE DES VARIABLES (XGBOOST)")
print("="*70)

# Obtenir l'importance des features
feature_importance = xgb_model.feature_importances_
feature_names = X.columns

# Créer un DataFrame
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print("\nTop 15 features par importance :")
print(importance_df.head(15).to_string(index=False))

In [ ]:
# Visualiser l'importance des features
fig, ax = plt.subplots(figsize=(10, 8))

# Top 15 features
top_features = importance_df.head(15)

ax.barh(range(len(top_features)), top_features['Importance'].values, color='steelblue', alpha=0.8)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'].values)
ax.set_xlabel('Importance Score', fontweight='bold', fontsize=11)
ax.set_title('Top 15 Features - XGBoost Baseline', fontweight='bold', fontsize=12)
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('plots/baseline_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n[OK] Graphique d'importance sauvegardé : plots/baseline_feature_importance.png")

## Étape 6 : SAUVEGARDE DU MODÈLE

In [ ]:
print("\n" + "="*70)
print("ÉTAPE 6 : SAUVEGARDE DES MODÈLES")
print("="*70)

# Sauvegarder le modèle XGBoost
xgb_model_path = 'models/modele_xgb_baseline.pkl'
with open(xgb_model_path, 'wb') as f:
    pickle.dump(xgb_model, f)

print(f"\n[OK] Modèle XGBoost sauvegardé")
print(f"    {xgb_model_path}")

# Sauvegarder le scaler
scaler_path = 'models/scaler_baseline.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f"\n[OK] StandardScaler sauvegardé")
print(f"    {scaler_path}")

# Sauvegarder le modèle Régression Linéaire
lr_model_path = 'models/modele_lr_baseline.pkl'
with open(lr_model_path, 'wb') as f:
    pickle.dump(lr_model, f)

print(f"\n[OK] Modèle Régression Linéaire sauvegardé")
print(f"    {lr_model_path}")

In [ ]:
# Sauvegarder les résultats des métriques
results_df = pd.DataFrame({
    'Modèle': ['Régression Linéaire', 'XGBoost'],
    'MAE_EUR': [metrics_lr_test['mae_euros'], metrics_xgb_test['mae_euros']],
    'RMSE_EUR': [metrics_lr_test['rmse_euros'], metrics_xgb_test['rmse_euros']],
    'MAPE_%': [metrics_lr_test['mape'], metrics_xgb_test['mape']],
    'R2_log': [metrics_lr_test['r2_log'], metrics_xgb_test['r2_log']]
})

results_path = 'results/baseline_metrics.csv'
results_df.to_csv(results_path, index=False, encoding='utf-8')

print(f"\n[OK] Résultats des métriques sauvegardés")
print(f"    {results_path}")

In [ ]:
# Résumé final
print("\n" + "="*70)
print("RÉSUMÉ FINAL")
print("="*70)

print(f"""
[OK] Modélisation baseline terminée!

Fichiers générés :
  1. modele_xgb_baseline.pkl       : Meilleur modèle (XGBoost)
  2. modele_lr_baseline.pkl        : Régression Linéaire
  3. scaler_baseline.pkl           : StandardScaler pour normalisation
  4. baseline_metrics.csv          : Résultats des performances
  5. baseline_feature_importance.png : Graphique importance des features

Performances finales (ensemble test) :
  
  Régression Linéaire :
    - MAE  : {metrics_lr_test['mae_euros']:,.0f} EUR/m²
    - RMSE : {metrics_lr_test['rmse_euros']:,.0f} EUR/m²
    - MAPE : {metrics_lr_test['mape']:.2f}%
    - R²   : {metrics_lr_test['r2_log']:.4f}
  
  XGBoost :
    - MAE  : {metrics_xgb_test['mae_euros']:,.0f} EUR/m²
    - RMSE : {metrics_xgb_test['rmse_euros']:,.0f} EUR/m²
    - MAPE : {metrics_xgb_test['mape']:.2f}%
    - R²   : {metrics_xgb_test['r2_log']:.4f}

Prochain pas : Optimiser les hyperparamètres (Notebook 4)
""")